# 05 - Tarea: Comparación e interpretación con datos Sophy/Piura

Esta tarea es la segunda entrega evaluada. Es una continuación natural de:

- `01_walkthrough_nowcasting_practico.ipynb`
- `02_tarea_nowcasting_metricas.ipynb`
- `03_comparacion_modelos_post_inferencia.ipynb`

A diferencia de los ejemplos guiados con IDEAM, esta tarea usa **únicamente datos Sophy/Piura**. El objetivo es aplicar el mismo enfoque de evaluación a un caso más cercano al contexto peruano.

**Duración estimada:** 2 a 4 horas  
**Dificultad:** similar a la tarea `02`  
**Entrega:** notebook ejecutado + interpretación breve

## Objetivo

Comparar predicciones de nowcasting sobre eventos de Piura/Sophy usando métricas básicas e interpretación meteorológica.

El estudiante debe responder:

> ¿La predicción mejora frente a persistencia?  
> ¿La mejora se observa en RMSE, en CSI, o en ambos?  
> ¿Qué ocurre con lluvia fuerte o intensa?

Si existen predicciones de EarthFormer o CasCast, se comparan contra persistencia. Si no existen, la tarea se puede completar usando persistencia como línea base y dejando explícita esa limitación.

## 0. Prerrequisitos

Antes de iniciar, verifica que existan datos Sophy/Piura. En el ambiente local del curso se encuentran en:

```text
/home/henryjuarez/code_andre_sophy/curso_hf/dataset/piura_data
```

También puedes colocar los archivos en:

```text
nowcasting_course_lab/data/piura_data/
```

Formato esperado:

```text
(T, H, W) = (25, 128, 128)
```

Convención:

```text
primeros 13 frames = input/contexto
últimos 12 frames = target/futuro
```

In [ ]:
from pathlib import Path
import sys
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from course_utils.palette import apply_course_style, RAIN_LEVELS
from course_utils.plotting import plot_event_grid, plot_target_prediction_panel, plot_rmse_bias, plot_csi_by_threshold

apply_course_style()

# Esta tarea usa unicamente datos Sophy/Piura.
# El notebook busca primero dentro del repo del curso y luego en la ruta local de trabajo.
PIURA_DATA_CANDIDATES = [
    PROJECT_ROOT / "data" / "piura_data",
    PROJECT_ROOT.parent / "curso_hf" / "dataset" / "piura_data",
    Path("/home/henryjuarez/code_andre_sophy/curso_hf/dataset/piura_data"),
]

PIURA_DATA_DIR = next((p for p in PIURA_DATA_CANDIDATES if p.exists()), None)
if PIURA_DATA_DIR is None:
    raise FileNotFoundError(
        "No encontre datos Piura/Sophy. Colocalos en data/piura_data/ "
        "o ajusta PIURA_DATA_DIR manualmente."
    )

PIURA_FILES = sorted([p.name for p in PIURA_DATA_DIR.glob("*.npy")])
LEAD_MINUTES = np.arange(5, 65, 5)
THRESHOLDS = [0.5, 2.0, 5.0, 10.0]
INPUT_FRAMES = 13
PRED_FRAMES = 12

PREDICTION_ROOT = PROJECT_ROOT / "outputs" / "predictions"
MODEL_DIRS = {
    "earthformer": PREDICTION_ROOT / "earthformer",
    "cascast": PREDICTION_ROOT / "cascast",
    "model": PREDICTION_ROOT / "model",
}

print("Directorio Piura/Sophy:", PIURA_DATA_DIR)
print("Numero de archivos Piura/Sophy:", len(PIURA_FILES))
print("Primeros archivos:", PIURA_FILES[:3])

## 1. Funciones de carga para Piura/Sophy

Completa o revisa estas funciones. La limpieza de NaN es importante porque algunas zonas pueden representar ausencia de eco o regiones fuera del radar.

In [ ]:
def clean_rain_array(array):
    """Convierte NaN/Inf a 0 y devuelve float32."""
    return np.nan_to_num(array.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)


def load_piura_sample(filename):
    path = PIURA_DATA_DIR / filename
    sequence = clean_rain_array(np.load(path))
    if sequence.ndim != 3 or sequence.shape[0] != 25:
        raise ValueError(f"Forma inesperada para {filename}: {sequence.shape}")
    return sequence


def split_sequence(sequence):
    inputs = sequence[:INPUT_FRAMES]
    target = sequence[INPUT_FRAMES:INPUT_FRAMES + PRED_FRAMES]
    return inputs, target


def make_persistence_prediction(inputs):
    """Repite el ultimo frame observado para los 12 tiempos futuros."""
    last_frame = inputs[-1]
    return np.repeat(last_frame[None, :, :], PRED_FRAMES, axis=0).astype(np.float32)


def parse_piura_datetime(filename):
    """Extrae fecha/hora desde nombres tipo piura_furuno_rr_seq_20250326_1400_len25_stride12.npy."""
    match = re.search(r"seq_(\d{8})_(\d{4})", filename)
    if match is None:
        return "unknown"
    date, hour = match.groups()
    return f"{date[:4]}-{date[4:6]}-{date[6:8]} {hour[:2]}:{hour[2:]}"

## 2. Auditoría rápida de los eventos Piura/Sophy

Construye una tabla simple para entender los casos antes de evaluar modelos.

Debe incluir:

- archivo;
- fecha/hora;
- máximo de lluvia;
- lluvia media;
- fracción de pixeles con lluvia `>= 0.5 mm/h`;
- fracción de pixeles con lluvia fuerte `>= 5 mm/h`.

In [ ]:
def build_piura_audit(files=PIURA_FILES):
    rows = []
    for filename in files:
        sequence = load_piura_sample(filename)
        rows.append({
            "filename": filename,
            "datetime": parse_piura_datetime(filename),
            "n_frames": sequence.shape[0],
            "H": sequence.shape[1],
            "W": sequence.shape[2],
            "max_rain_rate": float(np.nanmax(sequence)),
            "mean_rain_rate": float(np.nanmean(sequence)),
            "rainy_pixel_fraction": float((sequence >= 0.5).mean()),
            "heavy_rain_pixel_fraction": float((sequence >= 5.0).mean()),
        })
    return pd.DataFrame(rows)

piura_audit = build_piura_audit()
piura_audit

## 3. Elegir un caso y visualizarlo

Escoge un archivo de Piura/Sophy. Observa el movimiento de la precipitación y los núcleos intensos.

In [ ]:
sample_name = PIURA_FILES[0]
sequence = load_piura_sample(sample_name)
inputs, target = split_sequence(sequence)
persistence = make_persistence_prediction(inputs)

print("Caso:", sample_name)
print("Fecha/hora:", parse_piura_datetime(sample_name))
print("inputs:", inputs.shape, "target:", target.shape, "persistence:", persistence.shape)

fig = plot_event_grid(inputs, target, persistence, sample_name, "persistencia", lead_indices=(2, 5, 8, 11))
plt.show()

## 4. Métricas: reutilizar lo aprendido en la tarea 02

Aquí debes implementar o pegar las funciones que completaste en `02_tarea_nowcasting_metricas.ipynb`.

Funciones requeridas:

- `safe_pearson`
- `compute_continuous_metrics`
- `safe_divide`
- `compute_binary_metrics`
- `compute_event_metrics`

Esta tarea busca evaluar e interpretar, no introducir métricas mucho más difíciles.

In [ ]:
def safe_pearson(x, y):
    """TODO: correlación de Pearson segura entre dos campos 2D."""
    raise NotImplementedError("Completa o pega tu función de la tarea 02")


def compute_continuous_metrics(pred, target):
    """TODO: DataFrame con lead_min, MAE, RMSE, Bias, Pearson_r."""
    rows = []
    for i, lead in enumerate(LEAD_MINUTES):
        # TODO: error = pred[i] - target[i]
        # TODO: MAE, RMSE, Bias, Pearson_r
        raise NotImplementedError("Completa compute_continuous_metrics")
    return pd.DataFrame(rows)


def safe_divide(numerator, denominator):
    """TODO: devuelve np.nan si denominator == 0."""
    raise NotImplementedError("Completa safe_divide")


def compute_binary_metrics(pred, target, threshold):
    """TODO: TP, FP, FN, TN, CSI, POD, FAR, Precision, Recall, F1."""
    raise NotImplementedError("Completa compute_binary_metrics")


def compute_event_metrics(pred, target, thresholds=THRESHOLDS):
    """TODO: métricas por lead time y umbral."""
    rows = []
    for threshold in thresholds:
        for i, lead in enumerate(LEAD_MINUTES):
            # TODO: metrics = compute_binary_metrics(pred[i], target[i], threshold)
            # TODO: append threshold, lead y metrics
            raise NotImplementedError("Completa compute_event_metrics")
    return pd.DataFrame(rows)

## 5. Predicciones disponibles para Piura/Sophy

Esta función busca predicciones guardadas con el mismo nombre del archivo en:

```text
outputs/predictions/earthformer/
outputs/predictions/cascast/
outputs/predictions/model/
```

Si no existen, al menos se evalúa persistencia.

In [ ]:
def available_predictions_for_piura_case(filename):
    sequence = load_piura_sample(filename)
    inputs, target = split_sequence(sequence)
    predictions = {"persistence": make_persistence_prediction(inputs)}

    for model_name, folder in MODEL_DIRS.items():
        pred_path = folder / filename
        if pred_path.exists():
            predictions[model_name] = clean_rain_array(np.load(pred_path))

    return inputs, target, predictions

inputs, target, predictions = available_predictions_for_piura_case(sample_name)
print("Predicciones disponibles:", list(predictions))

## 6. Comparación visual por modelo

Compara los modelos disponibles para un mismo caso. Si solo hay persistencia, discute esa limitación.

In [ ]:
for model_name, pred in predictions.items():
    fig = plot_event_grid(inputs, target, pred, sample_name, model_name, lead_indices=(2, 5, 8, 11))
    plt.show()

## 7. Evaluación de un caso

Cuando completes tus funciones de métricas, calcula las tablas para cada modelo disponible.

In [ ]:
# Descomenta cuando tus funciones estén completas.
# case_rows = []
# for model_name, pred in predictions.items():
#     continuous_df = compute_continuous_metrics(pred, target)
#     event_df = compute_event_metrics(pred, target, THRESHOLDS)
#     case_rows.append({
#         "sample": sample_name,
#         "model": model_name,
#         "RMSE_mean": continuous_df["RMSE"].mean(),
#         "MAE_mean": continuous_df["MAE"].mean(),
#         "Bias_mean": continuous_df["Bias"].mean(),
#         "Pearson_mean": continuous_df["Pearson_r"].mean(),
#         "CSI_0.5_mean": event_df[event_df["threshold_mm_h"] == 0.5]["CSI"].mean(),
#         "CSI_5_mean": event_df[event_df["threshold_mm_h"] == 5.0]["CSI"].mean(),
#         "CSI_10_mean": event_df[event_df["threshold_mm_h"] == 10.0]["CSI"].mean(),
#     })
#
# case_summary = pd.DataFrame(case_rows)
# case_summary

## 8. Pipeline para todos los casos Piura/Sophy

Ahora aplica el mismo análisis a todos los archivos de Piura/Sophy.

Tabla requerida:

| sample | datetime | model | RMSE_mean | Bias_mean | CSI_0.5_mean | CSI_5_mean | CSI_10_mean |
|---|---|---:|---:|---:|---:|---:|

In [ ]:
def evaluate_all_piura_cases(files=PIURA_FILES):
    """TODO: evalúa todos los archivos Piura/Sophy y todos los modelos disponibles."""
    rows = []
    for filename in files:
        inputs, target, preds = available_predictions_for_piura_case(filename)
        for model_name, pred in preds.items():
            # TODO: continuous_df = compute_continuous_metrics(pred, target)
            # TODO: event_df = compute_event_metrics(pred, target, THRESHOLDS)
            # TODO: agregar fila resumen
            raise NotImplementedError("Completa evaluate_all_piura_cases")
    return pd.DataFrame(rows)

In [ ]:
# Descomenta cuando completes evaluate_all_piura_cases.
# piura_summary = evaluate_all_piura_cases()
# piura_summary

## 9. Análisis simple de extremos

Esta parte es una versión ligera del notebook avanzado `04`.

Calcula:

- máximo de lluvia observado por frame;
- máximo de lluvia predicho por frame;
- área con lluvia `>= 5 mm/h`;
- área con lluvia `>= 10 mm/h`.

Pregunta clave:

> ¿La predicción conserva los máximos de lluvia o los subestima?

In [ ]:
def simple_extreme_stats(sequence):
    """TODO: estadísticas simples de extremos por lead time."""
    rows = []
    for i, lead in enumerate(LEAD_MINUTES):
        frame = sequence[i]
        rows.append({
            "lead_min": int(lead),
            # TODO: max_rain_rate
            # TODO: area_ge_5
            # TODO: area_ge_10
        })
    return pd.DataFrame(rows)


def compare_target_pred_extremes(pred, target, model_name):
    """TODO: une extremos de target y predicción para graficar."""
    raise NotImplementedError("Completa compare_target_pred_extremes")

In [ ]:
# Ejemplo esperado cuando completes las funciones:
# for model_name, pred in predictions.items():
#     extremes_df = compare_target_pred_extremes(pred, target, model_name)
#     plt.scatter(extremes_df["target_max_rain_rate"], extremes_df["pred_max_rain_rate"], label=model_name)
# plt.plot([0, 60], [0, 60], "k--", linewidth=1)
# plt.xlabel("Máximo observado (mm/h)")
# plt.ylabel("Máximo predicho (mm/h)")
# plt.legend()
# plt.show()

## 10. Preguntas de interpretación

Responde en 300–500 palabras:

1. ¿La predicción supera a persistencia en Piura/Sophy?
2. ¿La mejora se ve más en RMSE o en CSI?
3. ¿Qué ocurre con lluvia fuerte o intensa (`5` y `10 mm/h`)?
4. ¿El modelo tiende a subestimar máximos?
5. ¿Hay evidencia de desplazamiento espacial?
6. ¿Qué limitaciones tiene este análisis?

Si solo evaluaste persistencia, explica qué resultados esperarías comparar cuando existan EarthFormer/CasCast.

## 11. Entregables

Entrega:

1. Notebook ejecutado.
2. Tabla de auditoría Piura/Sophy.
3. Figura visual de al menos un caso.
4. Tabla comparativa por archivo/modelo.
5. Gráfico simple de extremos.
6. Interpretación escrita de 300–500 palabras.

## Rúbrica sugerida

| Componente | Peso |
|---|---:|
| Carga y auditoría de datos Piura/Sophy | 15% |
| Reutilización/implementación correcta de métricas | 25% |
| Comparación por archivo/modelo | 25% |
| Análisis simple de extremos | 15% |
| Interpretación final | 20% |